# Colab 2 · Orchestration Patterns in Depth
### Day 19 — Agent Orchestration with AutoGen Studio & Semantic Kernel

Colab 1 used the simplest pattern (RoundRobin). Now you'll wire the **same research team four different ways** and watch how the *control flow* changes — then peek at the **same idea in Semantic Kernel**.

**You will build:**
1. **SelectorGroupChat** — an LLM decides who speaks next.
2. **Swarm** — agents hand off to each other directly.
3. **GraphFlow** — a deterministic researcher → writer → reviewer graph.
4. A **function tool** the researcher calls to delegate real work.
5. A **Semantic Kernel** sequential-orchestration mini-example.

⏱️ ~60 min including the extension tasks at the end.

> The patterns are the lesson. AutoGen, Semantic Kernel and the Microsoft Agent Framework all expose this same family — RoundRobin/Sequential, Selector/GroupChat, Swarm/Handoff, Graph, Magentic.

## 0 · Setup

In [1]:
%pip install -q -U "autogen-agentchat" "autogen-ext[openai]"
print("AutoGen installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.3/119.3 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.9/101.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 4.8 MB/s eta 0:00:00
AutoGen installed.


In [2]:
import os
from getpass import getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY")

from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.ui import Console

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
print("Ready.")

OPENAI_API_KEY··········
Ready.


### Three specialists we'll reuse

Notice the **descriptions** — Selector and Swarm route work based on them, so they have to be specific and non-overlapping.

In [3]:
def make_specialists():
    planner = AssistantAgent(
        name="planner",
        model_client=model_client,
        description="Breaks a topic into 2-3 concrete sub-questions to research.",
        system_message="You plan research. Given a topic, list 2-3 specific sub-questions. Keep it short.",
    )
    researcher = AssistantAgent(
        name="researcher",
        model_client=model_client,
        description="Answers factual sub-questions with concise bullet points.",
        system_message="You answer the planner's sub-questions with short factual bullets.",
    )
    writer = AssistantAgent(
        name="writer",
        model_client=model_client,
        description="Turns research bullets into a tight 4-sentence summary, ending with APPROVE.",
        system_message="Write a tight 4-sentence summary from the research. End your message with APPROVE.",
    )
    return planner, researcher, writer

print("Specialist factory ready.")

Specialist factory ready.


## 1 · SelectorGroupChat — let an LLM route

Instead of a fixed order, a **SelectorGroupChat** uses a model to pick *who should act next* based on the conversation and each agent's `description`. Good when the next best speaker depends on what just happened.

Key knobs: it needs its own `model_client` to do the routing, and `allow_repeated_speaker=False` stops one agent from monopolising the floor.

In [4]:
from autogen_agentchat.teams import SelectorGroupChat

planner, researcher, writer = make_specialists()
termination = TextMentionTermination("APPROVE") | MaxMessageTermination(8)

selector_team = SelectorGroupChat(
    participants=[planner, researcher, writer],
    model_client=model_client,        # the "router" brain
    termination_condition=termination,
    allow_repeated_speaker=False,
)

await Console(selector_team.run_stream(
    task="Topic: why are reusable cups better than disposable ones?"
))

---------- TextMessage (user) ----------
Topic: why are reusable cups better than disposable ones?
---------- TextMessage (planner) ----------
1. What are the environmental impacts of reusable cups compared to disposable cups in terms of resource consumption and waste generation?  
2. How do reusable cups contribute to reducing single-use plastic pollution?  
3. What are the cost implications for consumers and businesses when using reusable cups versus disposable ones?
---------- TextMessage (researcher) ----------
1. **Environmental Impacts**:  
   - Reusable cups require more resources to produce initially (materials, energy).  
   - Over time, reusable cups generate significantly less waste as they can be used multiple times.  
   - Disposable cups contribute to higher levels of waste, often ending up in landfills or as litter.  

2. **Reduction of Single-Use Plastic Pollution**:  
   - Reusable cups eliminate the need for single-use plastics by being a sustainable alternative.  
  

TaskResult(messages=[TextMessage(id='778f61c5-2ecd-4a47-8faf-8aceede155be', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 10, 59, 18, 829456, tzinfo=datetime.timezone.utc), content='Topic: why are reusable cups better than disposable ones?', type='TextMessage'), TextMessage(id='bc7caf6c-ad13-4fd3-9807-31058c654acb', source='planner', models_usage=RequestUsage(prompt_tokens=45, completion_tokens=58), metadata={}, created_at=datetime.datetime(2026, 6, 22, 10, 59, 23, 350614, tzinfo=datetime.timezone.utc), content='1. What are the environmental impacts of reusable cups compared to disposable cups in terms of resource consumption and waste generation?  \n2. How do reusable cups contribute to reducing single-use plastic pollution?  \n3. What are the cost implications for consumers and businesses when using reusable cups versus disposable ones?', type='TextMessage'), TextMessage(id='348bdc39-2614-443f-b07f-da6d514935a1', source='researcher', models_

Look at the speaker order in the transcript — it was **chosen at runtime**, not fixed. That's the difference from RoundRobin.

## 2 · Swarm — agents hand off to each other

In a **Swarm**, control is decentralised: each agent declares who it can **hand off** to via `handoffs=[...]`, and passes control with a handoff message. There's no central router — the agents themselves decide.

We'll build a tiny triage flow: a `triage` agent routes to either `billing` or `tech`, and those can hand back to the user when done.

In [5]:
from autogen_agentchat.teams import Swarm
from autogen_agentchat.conditions import HandoffTermination

triage = AssistantAgent(
    name="triage",
    model_client=model_client,
    handoffs=["billing", "tech"],
    description="Front desk: routes the user to the right specialist.",
    system_message="Decide if the request is about billing or tech, then hand off to that agent.",
)
billing = AssistantAgent(
    name="billing",
    model_client=model_client,
    handoffs=["triage"],
    description="Handles billing and refund questions.",
    system_message="Answer the billing question. If it's not billing, hand back to triage.",
)
tech = AssistantAgent(
    name="tech",
    model_client=model_client,
    handoffs=["triage"],
    description="Handles technical troubleshooting.",
    system_message="Answer the tech question concisely, then say DONE.",
)

swarm = Swarm(
    participants=[triage, billing, tech],          # Swarm starts with the first agent
    termination_condition=TextMentionTermination("DONE") | MaxMessageTermination(8),
)

await Console(swarm.run_stream(task="My app keeps crashing when I open the camera."))

---------- TextMessage (user) ----------
My app keeps crashing when I open the camera.
---------- ToolCallRequestEvent (triage) ----------
[FunctionCall(id='call_eU2IiFS1jhQaw7wFAYsE7oLi', arguments='{}', name='transfer_to_tech')]
---------- ToolCallExecutionEvent (triage) ----------
[FunctionExecutionResult(content='Transferred to tech, adopting the role of tech immediately.', name='transfer_to_tech', call_id='call_eU2IiFS1jhQaw7wFAYsE7oLi', is_error=False)]
---------- HandoffMessage (triage) ----------
Transferred to tech, adopting the role of tech immediately.
---------- ToolCallRequestEvent (tech) ----------
[FunctionCall(id='call_Xs77PBPIxaQ6Hkovo3cy3I4K', arguments='{}', name='transfer_to_triage')]
---------- ToolCallExecutionEvent (tech) ----------
[FunctionExecutionResult(content='Transferred to triage, adopting the role of triage immediately.', name='transfer_to_triage', call_id='call_Xs77PBPIxaQ6Hkovo3cy3I4K', is_error=False)]
---------- HandoffMessage (tech) ----------
Trans

TaskResult(messages=[TextMessage(id='607adf71-bf3f-4a8e-ad66-04e790bfd7a9', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 11, 1, 28, 572185, tzinfo=datetime.timezone.utc), content='My app keeps crashing when I open the camera.', type='TextMessage'), ToolCallRequestEvent(id='bae95a1d-03d0-48b6-8bcd-f0f6197c1daf', source='triage', models_usage=RequestUsage(prompt_tokens=82, completion_tokens=12), metadata={}, created_at=datetime.datetime(2026, 6, 22, 11, 1, 29, 906621, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_eU2IiFS1jhQaw7wFAYsE7oLi', arguments='{}', name='transfer_to_tech')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='eccb2602-1a51-485a-aae4-115380bbaf70', source='triage', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 11, 1, 29, 908576, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content='Transferred to tech, adopting the role of tech immediately.', name='transf

Watch for the **HandoffMessage** in the transcript — that's one agent explicitly delegating to another. `HandoffTermination(target="user")` is another common stop condition when an agent hands control back to a human.

## 3 · GraphFlow — a deterministic workflow

When you need the **same path every time** (auditable, reproducible), use **GraphFlow**. You declare nodes and directed edges with `DiGraphBuilder`; execution follows the graph exactly.

Here: `planner → researcher → writer`, a fixed pipeline with no LLM routing brain.

In [6]:
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow

planner, researcher, writer = make_specialists()

builder = DiGraphBuilder()
builder.add_node(planner).add_node(researcher).add_node(writer)
builder.add_edge(planner, researcher).add_edge(researcher, writer)
graph = builder.build()

flow = GraphFlow(
    participants=builder.get_participants(),
    graph=graph,
)

await Console(flow.run_stream(task="Topic: the benefits of cycling to work."))

---------- TextMessage (user) ----------
Topic: the benefits of cycling to work.
---------- TextMessage (planner) ----------
1. How does cycling to work impact employees' physical health compared to other commuting methods?  
2. What are the environmental benefits of increased cycling as a mode of transportation for commuters?  
3. How does cycling to work influence productivity and mental well-being in the workplace?
---------- TextMessage (researcher) ----------
1. **Impact on Physical Health:**
   - Increases cardiovascular fitness and overall stamina.
   - Reduces the risk of chronic illnesses (e.g., obesity, heart disease).
   - Lowers stress levels and enhances mood through exercise.
   - Helps in weight management and muscle toning.
   - Compared to driving or public transport, cycling incorporates physical activity into the daily routine.

2. **Environmental Benefits:**
   - Reduces greenhouse gas emissions contributing to climate change.
   - Decreases air pollution, particula

TaskResult(messages=[TextMessage(id='6d0a24f4-fdfc-4a4a-bd7b-291c644a2301', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 11, 16, 17, 144806, tzinfo=datetime.timezone.utc), content='Topic: the benefits of cycling to work.', type='TextMessage'), TextMessage(id='bb5ae9ba-c91f-40cf-abdd-8ef4ef59e975', source='planner', models_usage=RequestUsage(prompt_tokens=43, completion_tokens=55), metadata={}, created_at=datetime.datetime(2026, 6, 22, 11, 16, 18, 760138, tzinfo=datetime.timezone.utc), content="1. How does cycling to work impact employees' physical health compared to other commuting methods?  \n2. What are the environmental benefits of increased cycling as a mode of transportation for commuters?  \n3. How does cycling to work influence productivity and mental well-being in the workplace?", type='TextMessage'), TextMessage(id='9d63d9d6-8f10-40af-a6cd-a55cd78a6139', source='researcher', models_usage=RequestUsage(prompt_tokens=95, completion_toke

GraphFlow gives you **determinism**: the order is guaranteed by the graph, not decided by a model. That's exactly what you want for a compliance-sensitive or repeatable pipeline.

## 4 · A function tool the researcher can call

Delegation isn't only agent-to-agent — an agent can delegate to **code** via a tool. Define a plain Python function, pass it in `tools=[...]`, and the agent will call it when useful. (Here it's a stub; swap in a real search API at home.)

In [7]:
def web_search(query: str) -> str:
    """Look up a query and return a short text snippet. (Stub for the workshop.)"""
    canned = {
        "reusable cup co2": "A reusable cup typically breaks even vs. disposables after ~20-100 uses.",
        "default": "No exact match; returning a generic note that reusable goods amortise their footprint with use.",
    }
    return canned.get(query.lower().strip(), canned["default"])

researcher_with_tool = AssistantAgent(
    name="researcher",
    model_client=model_client,
    tools=[web_search],
    description="Researches facts, calling web_search when it needs evidence.",
    system_message="Use the web_search tool to find a figure, then report it in one bullet. End with APPROVE.",
)

from autogen_agentchat.teams import RoundRobinGroupChat
tool_team = RoundRobinGroupChat(
    [researcher_with_tool],
    termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(4),
)
await Console(tool_team.run_stream(task="Find a figure on reusable cup CO2 break-even and report it."))

---------- TextMessage (user) ----------
Find a figure on reusable cup CO2 break-even and report it.
---------- ToolCallRequestEvent (researcher) ----------
[FunctionCall(id='call_440ofr2naAD1HVqZw7cIWIV8', arguments='{"query":"reusable cup CO2 break-even figure"}', name='web_search')]
---------- ToolCallExecutionEvent (researcher) ----------
[FunctionExecutionResult(content='No exact match; returning a generic note that reusable goods amortise their footprint with use.', name='web_search', call_id='call_440ofr2naAD1HVqZw7cIWIV8', is_error=False)]
---------- ToolCallSummaryMessage (researcher) ----------
No exact match; returning a generic note that reusable goods amortise their footprint with use.
---------- TextMessage (researcher) ----------
- Reusable cups generally amortize their CO2 footprint with use, but specific figures can vary based on the material, usage frequency, and production processes of both reusable and disposable cups.

APPROVE


TaskResult(messages=[TextMessage(id='035d5b00-f017-44d7-ad9e-ec1e9ac4d34b', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 11, 18, 14, 475910, tzinfo=datetime.timezone.utc), content='Find a figure on reusable cup CO2 break-even and report it.', type='TextMessage'), ToolCallRequestEvent(id='9f679dfa-ab29-4874-a827-dbbbab293b4e', source='researcher', models_usage=RequestUsage(prompt_tokens=97, completion_tokens=21), metadata={}, created_at=datetime.datetime(2026, 6, 22, 11, 18, 15, 650839, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_440ofr2naAD1HVqZw7cIWIV8', arguments='{"query":"reusable cup CO2 break-even figure"}', name='web_search')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='e8651155-b77d-4673-bcd6-802b6d7c0f3d', source='researcher', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 11, 18, 15, 654064, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content='No exact ma

The transcript shows a **ToolCall** and its result — the agent delegated part of its job to your function. In production you'd scope tools to least privilege and validate their arguments.

## 5 · The same idea in Semantic Kernel

Semantic Kernel expresses these patterns too — its **Sequential** orchestration is the SK analogue of RoundRobin/GraphFlow-in-a-line. The code below shows the *shape* of SK agent orchestration.

> ⚠️ SK's agent-orchestration API is newer and evolving (and SK is in maintenance mode heading into the Microsoft Agent Framework). If an import path has moved, check the official Semantic Kernel docs — the **concept** is what transfers, not the exact symbol names.

In [8]:
%pip install -q -U semantic-kernel
print("Semantic Kernel installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 929.3/929.3 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.2/93.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.9/217.9 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.2/115.2 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.1/192.1 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.6/106.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [9]:
# The shape of an SK sequential orchestration: two agents, output of one feeds the next.
# Wrapped in try/except because SK's orchestration symbols move between versions.
import asyncio

try:
    from semantic_kernel.agents import ChatCompletionAgent
    from semantic_kernel.agents.orchestration.sequential import SequentialOrchestration
    from semantic_kernel.agents.runtime import InProcessRuntime
    from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion

    service = OpenAIChatCompletion(ai_model_id="gpt-4o-mini")

    sk_writer = ChatCompletionAgent(
        name="writer", service=service,
        instructions="Write one short paragraph on the given topic.",
    )
    sk_editor = ChatCompletionAgent(
        name="editor", service=service,
        instructions="Tighten the paragraph you receive into two crisp sentences.",
    )

    orchestration = SequentialOrchestration(members=[sk_writer, sk_editor])
    runtime = InProcessRuntime()
    runtime.start()

    result = await orchestration.invoke(
        task="The benefits of walking meetings.", runtime=runtime
    )
    print(await result.get())
    await runtime.stop_when_idle()

except Exception as e:
    print("SK orchestration symbols may have moved in your installed version.")
    print("Concept: members=[writer, editor] run in sequence, output -> input.")
    print("Check https://learn.microsoft.com/semantic-kernel for the current API.")
    print("Error was:", type(e).__name__, e)

Walking meetings provide a refreshing alternative to traditional discussions by promoting physical activity and enhancing mental clarity. This approach fosters open communication and creativity while contributing to improved health and employee satisfaction.


Notice the **identical mental model**: a list of agents, run in order, each one's output feeding the next. RoundRobin (AutoGen) ≈ Sequential (SK) ≈ a linear GraphFlow. Learn it once.

---
## 🚀 Extension tasks

### Extension 1 — Write a custom selector function
`SelectorGroupChat` accepts a `selector_func` that overrides the LLM router with your own logic. Write a function that **forces** `planner` to go first, then lets the model choose. (Signature: it receives the message history and returns the next speaker's name, or `None` to defer to the model.)

### Extension 2 — Add a conditional GraphFlow edge
Extend the graph from §3 with a **reviewer** node and a **conditional edge**: if the writer's output contains the word `REVISE`, loop back to the writer; otherwise finish. Use `DiGraphBuilder`'s conditional-edge support and a stop condition so it can't loop forever.

### Extension 3 — Nest a team inside a graph node
A node in a GraphFlow can itself be a **team**. Replace the single `researcher` node with a 2-agent `RoundRobinGroupChat` (researcher + fact-checker) and wire that team in as one node. This is *composition*: patterns nest inside patterns.

Scaffolds below.

In [ ]:
# === Extension 1 scaffold: custom selector_func ===
# def force_planner_first(messages):
#     # Return an agent name (str) to force a speaker, or None to let the model decide.
#     if len(messages) <= 1:
#         return "planner"
#     return None
#
# planner, researcher, writer = make_specialists()
# team = SelectorGroupChat(
#     participants=[planner, researcher, writer],
#     model_client=model_client,
#     selector_func=force_planner_first,
#     termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(8),
# )
# await Console(team.run_stream(task="Topic: benefits of a standing desk."))

In [ ]:
# === Extension 2 scaffold: conditional edge (writer loops on REVISE) ===
# builder = DiGraphBuilder()
# builder.add_node(planner).add_node(researcher).add_node(writer).add_node(reviewer)
# builder.add_edge(planner, researcher).add_edge(researcher, writer).add_edge(writer, reviewer)
# # Conditional: reviewer -> writer only if "REVISE" appears; else the run ends.
# builder.add_edge(reviewer, writer, condition=lambda msg: "REVISE" in msg.to_text())
# graph = builder.build()
# flow = GraphFlow(participants=builder.get_participants(), graph=graph)

In [ ]:
# === Extension 3 scaffold: nest a team inside a node ===
# research_team = RoundRobinGroupChat(
#     [researcher, fact_checker],
#     termination_condition=MaxMessageTermination(4),
# )
# builder = DiGraphBuilder()
# builder.add_node(planner).add_node(research_team).add_node(writer)
# builder.add_edge(planner, research_team).add_edge(research_team, writer)

In [10]:
# ============================================================
# Extension 1 — Custom selector_func
# Force planner to go first, then defer to the model router.
# ============================================================

from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.ui import Console

# --- Custom selector function ---
def force_planner_first(messages):
    """
    SelectorGroupChat calls this function every time it needs to decide
    who should speak next.

    Parameters
    ----------
    messages : list
        The full message history so far in the team conversation.
        This includes the initial user task as the first message.

    Returns
    -------
    str | None
        - Return an agent name (e.g. "planner") to force that speaker.
        - Return None to let the built-in LLM router choose.
    """

    # At the very beginning of the conversation, only the user's task
    # is in the message history. So if there's just 1 message, we know
    # no agent has spoken yet.
    if len(messages) == 1:
        return "planner"   # force planner to speak first

    # After planner has gone first, return None so SelectorGroupChat
    # falls back to its normal model-based routing logic.
    return None


# Recreate the three specialist agents from your helper
planner, researcher, writer = make_specialists()

# Stop when writer says APPROVE, or after max 8 total messages
termination = TextMentionTermination("APPROVE") | MaxMessageTermination(8)

# Build the SelectorGroupChat
selector_team = SelectorGroupChat(
    participants=[planner, researcher, writer],
    model_client=model_client,              # router LLM used when selector_func returns None
    selector_func=force_planner_first,      # our override hook
    termination_condition=termination,
    allow_repeated_speaker=False,           # avoid one agent speaking repeatedly
)

# Run it
await Console(
    selector_team.run_stream(
        task="Topic: benefits of a standing desk."
    )
)

---------- TextMessage (user) ----------
Topic: benefits of a standing desk.
---------- TextMessage (planner) ----------
1. What impact does using a standing desk have on employee productivity and focus?
2. How does a standing desk influence physical health outcomes, such as posture and back pain?
3. What are the psychological effects of using a standing desk compared to traditional seating arrangements?
---------- TextMessage (researcher) ----------
1. Impact on Employee Productivity and Focus:
   - Increased energy levels lead to improved alertness.
   - Potential for reduced fatigue during work hours.
   - Ability to engage more actively in tasks, promoting greater focus.
   - Some studies report a slight increase in productivity scores.

2. Influence on Physical Health Outcomes:
   - Promotes better posture by encouraging alignment.
   - May reduce the risk and severity of back pain.
   - Supports healthier blood circulation.
   - Can help in burning more calories compared to sitti

TaskResult(messages=[TextMessage(id='d7a2a5d2-c0db-4237-93a8-9a0519e023d3', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 11, 39, 48, 801161, tzinfo=datetime.timezone.utc), content='Topic: benefits of a standing desk.', type='TextMessage'), TextMessage(id='ee99560f-ccfd-41bc-a4a3-c41d54401553', source='planner', models_usage=RequestUsage(prompt_tokens=42, completion_tokens=53), metadata={}, created_at=datetime.datetime(2026, 6, 22, 11, 39, 50, 795676, tzinfo=datetime.timezone.utc), content='1. What impact does using a standing desk have on employee productivity and focus?\n2. How does a standing desk influence physical health outcomes, such as posture and back pain?\n3. What are the psychological effects of using a standing desk compared to traditional seating arrangements?', type='TextMessage'), TextMessage(id='963a9abd-ef52-43d8-b1b3-7b38003c4dc1', source='researcher', models_usage=RequestUsage(prompt_tokens=92, completion_tokens=157), metad

In [15]:
# ============================================================
# Extension 3 — WORKING VERSION for current AutoGen build
# ============================================================

#
# we flatten the nested team into two graph nodes:
#     planner -> researcher -> fact_checker -> writer
#
# This preserves the intended behavior:
# - planner creates sub-questions
# - researcher answers them
# - fact_checker reviews/improves the research
# - writer writes the final summary
# ============================================================

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow
from autogen_agentchat.ui import Console

# ------------------------------------------------------------
# 1) Define agents
# ------------------------------------------------------------
planner = AssistantAgent(
    name="planner",
    model_client=model_client,
    description="Breaks a topic into 2-3 concrete sub-questions to research.",
    system_message=(
        "You plan research. Given a topic, list 2-3 specific sub-questions. Keep it short."
    ),
)

researcher = AssistantAgent(
    name="researcher",
    model_client=model_client,
    description="Answers the planner's sub-questions with concise factual bullet points.",
    system_message=(
        "You are the researcher. Answer the planner's sub-questions with short factual bullet points. "
        "Do not write the final summary."
    ),
)

fact_checker = AssistantAgent(
    name="fact_checker",
    model_client=model_client,
    description="Checks the researcher's bullets for accuracy, clarity, and weak claims.",
    system_message=(
        "You are a fact-checker. Review the researcher's bullets for accuracy and clarity. "
        "Correct vague or weak claims and produce a cleaned-up bullet list for the writer."
    ),
)

writer = AssistantAgent(
    name="writer",
    model_client=model_client,
    description="Turns research bullets into a tight 4-sentence summary ending with APPROVE.",
    system_message=(
        "Write a tight 4-sentence summary from the research you receive. End your message with APPROVE."
    ),
)

# ------------------------------------------------------------
# 2) Build graph
# ------------------------------------------------------------
# Flattened version of the nested team:
#
# planner -> researcher -> fact_checker -> writer
# ------------------------------------------------------------
builder = DiGraphBuilder()
builder.add_node(planner)
builder.add_node(researcher)
builder.add_node(fact_checker)
builder.add_node(writer)

builder.add_edge(planner, researcher)
builder.add_edge(researcher, fact_checker)
builder.add_edge(fact_checker, writer)

graph = builder.build()

# ------------------------------------------------------------
# 3) Create GraphFlow
# ------------------------------------------------------------
flow = GraphFlow(
    participants=builder.get_participants(),
    graph=graph,
)

# ------------------------------------------------------------
# 4) Run it
# ------------------------------------------------------------
await Console(
    flow.run_stream(
        task="Topic: why public parks matter in cities."
    )
)

---------- TextMessage (user) ----------
Topic: why public parks matter in cities.
---------- TextMessage (planner) ----------
1. How do public parks impact the physical and mental health of urban residents?
2. What role do public parks play in enhancing community cohesion and social interaction?
3. How do green spaces in cities contribute to environmental sustainability and biodiversity?
---------- TextMessage (researcher) ----------
1. **Impact on Physical and Mental Health:**
   - Provide spaces for exercise, promoting physical activity, which can lead to reduced obesity rates.
   - Offer areas for relaxation and stress relief, contributing to improved mental well-being.
   - Access to nature is associated with lower levels of anxiety and depression.
   - Parks often provide venues for recreational activities, such as sports and walking, enhancing overall fitness.

2. **Role in Community Cohesion and Social Interaction:**
   - Serve as gathering places for events, fostering a sense 

TaskResult(messages=[TextMessage(id='84dbc100-d57a-4009-b816-1d60ae28d571', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 11, 47, 38, 228912, tzinfo=datetime.timezone.utc), content='Topic: why public parks matter in cities.', type='TextMessage'), TextMessage(id='d0979851-072e-4a5c-9d6e-9a0025eeaffc', source='planner', models_usage=RequestUsage(prompt_tokens=43, completion_tokens=47), metadata={}, created_at=datetime.datetime(2026, 6, 22, 11, 47, 39, 877886, tzinfo=datetime.timezone.utc), content='1. How do public parks impact the physical and mental health of urban residents?\n2. What role do public parks play in enhancing community cohesion and social interaction?\n3. How do green spaces in cities contribute to environmental sustainability and biodiversity?', type='TextMessage'), TextMessage(id='043e3b21-165e-4a4b-a16a-3f466aeb6279', source='researcher', models_usage=RequestUsage(prompt_tokens=99, completion_tokens=228), metadata={}, created_

## Recap

You orchestrated one research team **five ways** and saw exactly how control flow differs:

* **Selector** — LLM picks the next speaker (dynamic).
* **Swarm** — agents hand off to each other (decentralised).
* **GraphFlow** — a fixed, auditable graph (deterministic).
* **Tools** — an agent delegates work to a function.
* **Semantic Kernel** — the same Sequential idea in the enterprise SDK.

Pair this with the decision matrix from the slides, then tackle the **capstone**: build your own delegating team in AutoGen Studio.

In [ ]:
await model_client.close()
print("Client closed. On to the capstone!")